# Debut of Quantitative Research

In [ ]:
import matplotlib.pyplot as plt

params = {'legend.fontsize': 'x-large',
      'figure.figsize': (12, 6),
      'axes.labelsize': 'x-large',
      'axes.titlesize': 'x-large',
      'xtick.labelsize': 'x-large',
      'ytick.labelsize': 'x-large'}
plt.rcParams.update(params)

## Package: ffn
* Official document: https://pmorissette.github.io/ffn/

In [ ]:
!pip install ffn

### Example 1: Fetching Daily Price of Single Asset

#### Data Access

In [ ]:
import ffn

tbl = ffn.get("2330.tw", start = "2006-01-01")

#### Plot

In [ ]:
%matplotlib inline

tbl.plot.line(grid = True, figsize = (12, 6))

#### Plot After Rebasing

In [ ]:
tbl.rebase().plot.line(grid = True, figsize = (12, 6))

#### Calculation of Return Rate: pct_change()

In [ ]:
return_rates = tbl.pct_change() * 100
return_rates.plot.line(grid = True, figsize = (12, 6))

In [ ]:
return_rates.plot.hist(bins = 30, alpha = 0.3, grid = True, figsize = (12, 6))

#### Statistics Summary

In [ ]:
tbl.calc_stats().display()

### Example 2: International Market

In [ ]:
tbl2 = ffn.get("^TWII, ^N225, ^GSPC, ^GDAXI, ^HSI", start = "2006-01-01")

In [ ]:
tbl2.rebase().plot(grid = True, figsize = (12, 6))

In [ ]:
tbl2.calc_stats().display()

#### Correlation

In [ ]:
tbl2.pct_change().corr()

In [ ]:
tbl2.pct_change().plot_corr_heatmap(figsize = (6, 6))

### Example 3: Signal Generation & Backtesting

In [ ]:
from ffn.utils import clean_ticker

target = "2330.tw"
name = clean_ticker(target)
asset = ffn.get(target, start = "2020-01-01")

asset

#### Signal Generation

In [ ]:
asset["SMA5"] = asset[name].rolling(5).mean()
asset["SMA10"] = asset[name].rolling(10).mean()

asset

In [ ]:
buy_signal_mask = (asset["SMA5"].shift(2) < asset["SMA10"].shift(2)) & (asset["SMA5"].shift(1) > asset["SMA10"].shift(1))
buy_signal_mask

In [ ]:
sell_signal_mask = (asset["SMA5"].shift(2) > asset["SMA10"].shift(2)) & (asset["SMA5"].shift(1) < asset["SMA10"].shift(1))

#### Plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize = (12, 6))
plt.plot(asset[name], ".-", markersize = 8)
plt.plot(asset["SMA5"], linestyle = "--")
plt.plot(asset["SMA10"], linestyle = "--")
plt.plot(asset[name][buy_signal_mask], "r^", markersize = 12)
plt.plot(asset[name][sell_signal_mask], "gv", markersize = 12)
plt.grid(True)
plt.legend(["Close price", "SMA5", "SMA10"])

#### Backtesting: Buy Only

In [ ]:
position = False
asset["PV"] = 0.0
turnovers = 0

for i, t in enumerate(asset.index):
    
    if not position:
        
        payoff = 0
        asset["PV"].iloc[i] = asset["PV"].iloc[i - 1] + payoff
        
        if buy_signal_mask[t]:
            position = True
            turnovers = turnovers + 1
            print(">" * 5, "Create a long position.")
    else:
        
        payoff = asset[name].iloc[i] - asset[name].iloc[i - 1]
        asset["PV"].iloc[i] = asset["PV"].iloc[i - 1] + payoff
       
        if sell_signal_mask[t]:
            position = False
            print("<" * 5, "Close a long position.")
            print("{} -> {:.2f} -> {:.2f}".format(t, asset[name][t], asset["PV"][t]))

print("Number of turnovers:", turnovers)

#### Performance Report

##### Max Drawdown (MDD)
- https://en.wikipedia.org/wiki/Drawdown_(economics)

In [ ]:
asset["MDD (Ours)"] = (asset["PV"] - asset["PV"].cummax()).cummin()
dd_idx = asset["MDD (Ours)"].idxmin()
asset["Buy&Hold"] = asset[name].diff().cumsum()
asset["MDD (Buy&Hold)"] = (asset["Buy&Hold"] - asset["Buy&Hold"].cummax()).cummin()
dd_idx2 = asset["MDD (Buy&Hold)"].idxmin()

##### Benchmark

In [ ]:
asset[["PV", "Buy&Hold"]].plot(style = ".-", grid = True, figsize = (12, 6))
ax = asset["MDD (Ours)"].plot(style = "--", grid = True)
ax = asset["MDD (Buy&Hold)"].plot(style = "--", grid = True)
ax.legend(["Our strategy", "Benchmark: Buy&Hold", "MDD (Ours)", "MDD (Buy&Hold)"])

### Example 4: Extension to Example 3
* Modify Example 3 so that the program can allow short positions.

In [ ]:
position = False
asset["PV"] = 0.0
turnovers = 0

for i, t in enumerate(asset.index):
    
    if not position:
        
        payoff = 0
        
        if buy_signal_mask[t]:
            position = True
            turnovers = turnovers + 1
            print(">" * 5, "Create a long position.")
            
        if sell_signal_mask[t]:
            position = True
            turnovers = turnovers + 1
            print(">" * 5, "Create a short position.")
    else:
        
        payoff = asset[name].iloc[i] - asset[name].iloc[i - 1]
       
        if sell_signal_mask[t]:
            position = False
            print("<" * 5, "Close a long position.")
            
        if buy_signal_mask[t]:
            position = False
            payoff = -payoff
            print("<" * 5, "Close a short position.")
            
            
    asset["PV"].iloc[i] = asset["PV"].iloc[i - 1] + payoff
    print("{} -> {:.2f} -> {:.2f}".format(t, asset[name][t], asset["PV"][t]))

print("Number of turnovers", turnovers)

In [ ]:
asset["MDD (Ours)"] = (asset["PV"] - asset["PV"].cummax()).cummin()
dd_idx = asset["MDD (Ours)"].idxmin()
asset["Buy&Hold"] = asset[name].diff().cumsum()
asset["MDD (Buy&Hold)"] = (asset["Buy&Hold"] - asset["Buy&Hold"].cummax()).cummin()
dd_idx2 = asset["MDD (Buy&Hold)"].idxmin()
asset[["PV", "Buy&Hold"]].plot(style = ".-", grid = True, figsize = (12, 6))
ax = asset["MDD (Ours)"].plot(style = "--", grid = True)
ax = asset["MDD (Buy&Hold)"].plot(style = "--", grid = True)
ax.legend(["Our strategy", "Benchmark: Buy&Hold", "MDD (Ours)", "MDD (Buy&Hold)"])

### What You Can Do
* Change the target asset: https://www.tej.com.tw/webtej/doc/uid.htm
* Modify the program so that it could be used for mulit-asset strategies.